In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#%%
import os
from geopandas import read_parquet,points_from_xy
from shapely import Polygon
import geopandas as gpd
import pandas as pd
import numpy as np
import scanpy as sc
import pyarrow.parquet as pq
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt

# Evaluation on synthetic datasets

In [ ]:
# directory paths to model outputs
MISTIC_RESULTS_PATH = '../data/normal_mistic'
ABLATED_RESULTS_PATH = '../data/spatial_ablated'
RESOLVI_RESULTS_PATH = '../data/resolvi_results'
MERSCOPE_DATADIR = '../data/merscope_hcc1'
OUTPUT_DATADIR = '../data'

In [ ]:
# calculation of positive control and negative control delta expression
def pos_neg(number_tx_removed, ground_truth_pos, ground_truth_neg):
    number_tx_removed = number_tx_removed.copy()
    number_tx_removed.index = pd.MultiIndex.from_arrays(number_tx_removed[['cell_id', 'gene']].values.T,
                                                    names=['cell_id', "gene"])
    number_tx_removed.drop(columns=['cell_id', "gene"], inplace=True)
    positive_result = ground_truth_pos.merge(number_tx_removed, 
                        how='left', left_index=True, right_index=True).rename(columns={"molecule_id": "n_truth",
                                                                                        "value": "n_removed"})
    positive_result['n_removed'] = positive_result['n_removed'].fillna(0)
    positive_result["correction"] = positive_result["n_removed"]/positive_result['n_truth']

    negative_result = ground_truth_neg.merge(number_tx_removed, 
                        how='left', left_index=True, right_index=True).rename(columns={"molecule_id": "n_truth",
                                                                                        "value": "n_removed"})
    negative_result['n_removed'] = negative_result['n_removed'].fillna(0)
    negative_result["correction"] = negative_result["n_removed"]/negative_result['n_truth']
    
    return positive_result, negative_result

In [ ]:
# iterate through synthetic datasets, load the relevant saved files for each method, and calculate MAE scores
datasets = ['tx05', 'tx10', 'tx15', 'tx20', 'tx25', 'tx30']
methods_in_use = ['mistic', 'ablated', 'resolvi']
benchmark_results = []

for ds in tqdm(datasets):
    # get synthetic transcripts
    synthetic_tx_metadata = pd.read_csv(os.path.join(MERSCOPE_DATADIR, f'synthetic_{ds}_transcript_meta.csv')).rename(
        columns={'Unnamed: 0': 'molecule_id'}
    )
    cell_by_gene_counts = pd.pivot_table(synthetic_tx_metadata, index='cell_id', columns='gene', aggfunc='count', values='molecule_id')

    poscons = synthetic_tx_metadata[synthetic_tx_metadata['tx_type'] == 'synthetic_pos']
    poscons = poscons.groupby(['cell_id', 'gene'], as_index=False)['molecule_id'].count()
    poscons.index = pd.MultiIndex.from_arrays(poscons[['cell_id', 'gene']].values.T, names=['cell_id', "gene"])
    poscons.drop(columns=['cell_id', "gene"], inplace=True)

    negcons = synthetic_tx_metadata[synthetic_tx_metadata['tx_type'] == 'synthetic_neg']
    negcons = negcons.groupby(['cell_id', 'gene'], as_index=False)['molecule_id'].count()
    negcons.index = pd.MultiIndex.from_arrays(negcons[['cell_id', 'gene']].values.T, names=['cell_id', "gene"])

    if 'mistic' in methods_in_use:
        # get mistic results
        reassignments = pd.read_parquet(os.path.join(MISTIC_RESULTS_PATH, f"{ds}/mistic_tx_to_reassign.parquet"))
        drops = pd.read_parquet(os.path.join(MISTIC_RESULTS_PATH, f"{ds}/mistic_tx_to_remove.parquet"))

        # group mistic results by cell id
        number_tx_reassigned = reassignments.groupby(['from_cell_id', 'gene'], as_index=False)['molecule_id'].count().rename(
            columns={"from_cell_id": "cell_id", "molecule_id": "value"}
        )
        number_tx_dropped = drops.groupby(['from_cell_id', 'gene'], as_index=False)['molecule_id'].count().rename(
            columns={"from_cell_id": "cell_id", "molecule_id": "value"}
        )
        number_tx_corrected = pd.concat([reassignments, drops], axis=0).groupby(
            ['from_cell_id', 'gene'], as_index=False
        )['molecule_id'].count().rename(
            columns={"from_cell_id": "cell_id", "molecule_id": "value"}
        )

        # calculate mae values
        mistic_pos, mistic_neg = pos_neg(number_tx_corrected, poscons, negcons)   
        pos_mae = (mistic_pos['correction']-1).abs().mean()
        neg_mae = (mistic_neg['correction']-0).abs().mean() 

        # collate results
        benchmark_results.append([ds, 'mistic', pos_mae, neg_mae])

    if 'ablated' in methods_in_use:
        # get mistic results
        reassignments = pd.read_parquet(os.path.join(ABLATED_RESULTS_PATH, f"{ds}/mistica_tx_to_reassign.parquet"))
        drops = pd.read_parquet(os.path.join(ABLATED_RESULTS_PATH, f"{ds}/mistica_tx_to_remove.parquet"))

        # group mistic results by cell id
        number_tx_reassigned = reassignments.groupby(['from_cell_id', 'gene'], as_index=False)['molecule_id'].count().rename(
            columns={"from_cell_id": "cell_id", "molecule_id": "value"}
        )
        number_tx_dropped = drops.groupby(['from_cell_id', 'gene'], as_index=False)['molecule_id'].count().rename(
            columns={"from_cell_id": "cell_id", "molecule_id": "value"}
        )
        number_tx_corrected = pd.concat([reassignments, drops], axis=0).groupby(
            ['from_cell_id', 'gene'], as_index=False
        )['molecule_id'].count().rename(
            columns={"from_cell_id": "cell_id", "molecule_id": "value"}
        )

        # calculate mae values
        mistic_pos, mistic_neg = pos_neg(number_tx_corrected, poscons, negcons)   
        pos_mae = (mistic_pos['correction']-1).abs().mean()
        neg_mae = (mistic_neg['correction']-0).abs().mean() 

        # collate results
        benchmark_results.append([ds, 'ablated', pos_mae, neg_mae])

    if 'resolvi' in methods_in_use:
        # get resolvi_results 
        resolvi_adata = sc.read_h5ad(os.path.join(RESOLVI_RESULTS_PATH, f'synthetic_{ds}_supervised_resolvi.h5ad'))
        resolvi_result = pd.DataFrame(resolvi_adata.layers['generated_expression'], index=resolvi_adata.obs_names, columns=resolvi_adata.var_names)

        number_tx_removed = cell_by_gene_counts.reindex_like(resolvi_result).fillna(0) - resolvi_result
        number_tx_removed = pd.melt(number_tx_removed, ignore_index=False, var_name="gene").reset_index()

        # calculate mae values
        resolvi_pos, resolvi_neg = pos_neg(number_tx_removed=number_tx_removed, ground_truth_pos=poscons, ground_truth_neg=negcons)
        pos_mae = (resolvi_pos['correction']-1).abs().mean()
        neg_mae = (resolvi_neg['correction']-0).abs().mean() 

        # collate results
        benchmark_results.append([ds, 'resolvi', pos_mae, neg_mae])
    

mistic_benchmark_df = pd.DataFrame(benchmark_results, columns=['dataset', 'method', 'pos_mae', 'neg_mae'])
mistic_benchmark_df.to_csv(os.path.join(OUTPUT_DATADIR, 'methods_benchmark_mae.csv'), index=False)

In [ ]:
# plot results
fig, axs = plt.subplots(1, 2, figsize=(8, 3))

mistic_benchmark_df = pd.read_csv(os.path.join(OUTPUT_DATADIR, 'methods_benchmark_mae.csv'))
mistic_benchmark_df = mistic_benchmark_df.replace({
    'mistic': 'MisTIC', 'ablated': 'Ablated', 'resolvi': 'ResolVI',
    'tx05': '5', 'tx10': '10', 'tx15': '15', 'tx20': '20', 'tx25': '25', 'tx30': '30'
})
sns.barplot(
    mistic_benchmark_df,
    x='dataset', y='pos_mae', hue='method',
    legend=False, 
    ax=axs[0]
)
axs[0].set_ylabel('Positive control MAE')
axs[0].set_xlabel('Synthetic transcripts per cell')

sns.barplot(
    mistic_benchmark_df,
    x='dataset', y='neg_mae', hue='method',
    legend=True, 
    ax=axs[1]
)
axs[1].set_ylabel('Negative control MAE')
axs[1].set_xlabel('Synthetic transcripts per cell')
plt.legend(loc='upper right', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig('../results/fig2_benchmarking.png')

# Alternative simulations

In [ ]:
alternative_simulation_tests = []
ds = 'tx05'

MISTIC_RESULTS_PATH = '../data/normal_mistic'
CALIBRATION_RESULTS_PATH = '../data/updated_threshold/synthetic3_original_thresh_1_3'
MERSCOPE_DATADIR = '../data/merscope_hcc1'
OUTPUT_DATADIR = '../data'

model_to_specs = { # synthetic input, reassign results table, remove results table, probability of spillover
    'mistic': (os.path.join(MERSCOPE_DATADIR, 'synthetic_tx05_transcript_meta.csv'), 
               os.path.join(MISTIC_RESULTS_PATH, "tx05/mistic_tx_to_reassign.parquet"),
               os.path.join(MISTIC_RESULTS_PATH, "tx05/mistic_tx_to_remove.parquet"),
               0.5),
    'low_spillover': (os.path.join(MERSCOPE_DATADIR, 'synthetic_tx05_doublet0.05_transcript_meta.csv'), 
                      os.path.join(MISTIC_RESULTS_PATH, "tx05_low_doublet/tx05_low_doublet_tx_to_reassign.parquet"),
                      os.path.join(MISTIC_RESULTS_PATH, "tx05_low_doublet/tx05_low_doublet_tx_to_reassign.parquet"),
                      0.05),
    'high_spillover': (os.path.join(MERSCOPE_DATADIR, 'synthetic_tx05_doublet0.95_transcript_meta.csv'), 
                       os.path.join(MISTIC_RESULTS_PATH, "tx05_high_doublet/tx05_high_doublet_tx_to_reassign.parquet"),
                       os.path.join(MISTIC_RESULTS_PATH, "tx05_high_doublet/tx05_hight_doublet_tx_to_reassign.parquet"),
                       0.95),
    'resolvi_calibrated': (os.path.join(MERSCOPE_DATADIR, 'synthetic_tx05_doublet0.95_transcript_meta.csv'), 
                           os.path.join(CALIBRATION_RESULTS_PATH, f"tx05/mistic_tx_to_reassign.parquet"),
                           os.path.join(CALIBRATION_RESULTS_PATH, f"tx05/mistic_tx_to_remove.parquet"),
                           0.5),
}

for model_name in model_to_specs:
    infile, reassign_file, remove_file, p_spill = model_to_specs[model_name]
    # load ground truth data
    synthetic_tx_metadata = pd.read_csv(infile).rename(
            columns={'Unnamed: 0': 'molecule_id'}
        )
    poscons = synthetic_tx_metadata[synthetic_tx_metadata['tx_type'] == 'synthetic_pos']
    poscons = poscons.groupby(['cell_id', 'gene'], as_index=False)['molecule_id'].count()
    poscons.index = pd.MultiIndex.from_arrays(poscons[['cell_id', 'gene']].values.T, names=['cell_id', "gene"])
    poscons.drop(columns=['cell_id', "gene"], inplace=True)

    negcons = synthetic_tx_metadata[synthetic_tx_metadata['tx_type'] == 'synthetic_neg']
    negcons = negcons.groupby(['cell_id', 'gene'], as_index=False)['molecule_id'].count()
    negcons.index = pd.MultiIndex.from_arrays(negcons[['cell_id', 'gene']].values.T, names=['cell_id', "gene"])

    # get mistic results
    reassignments = pd.read_parquet(reassign_file)
    drops = pd.read_parquet(remove_file)

    # group mistic results by cell id
    number_tx_reassigned = reassignments.groupby(['from_cell_id', 'gene'], as_index=False)['molecule_id'].count().rename(
        columns={"from_cell_id": "cell_id", "molecule_id": "value"}
    )
    number_tx_dropped = drops.groupby(['from_cell_id', 'gene'], as_index=False)['molecule_id'].count().rename(
        columns={"from_cell_id": "cell_id", "molecule_id": "value"}
    )
    number_tx_corrected = pd.concat([reassignments, drops], axis=0).groupby(
        ['from_cell_id', 'gene'], as_index=False
    )['molecule_id'].count().rename(
        columns={"from_cell_id": "cell_id", "molecule_id": "value"}
    )

    # calculate mae values
    mistic_pos, mistic_neg = pos_neg(number_tx_corrected, poscons, negcons)   
    pos_mae = (mistic_pos['correction']-1).abs().mean()
    neg_mae = (mistic_neg['correction']-0).abs().mean() 

    alternative_simulation_tests.append([ds, model_name, p_spill, pos_mae, neg_mae])
    
alternative_benchmarking_df = pd.DataFrame(
    alternative_simulation_tests,
    columns=['dataset', 'variant', 'spillover_fraction', 'pos_mae', 'neg_mae']
)
alternative_benchmarking_df.to_csv('../data/alternative_mistic_benchmarks.csv', index=False)

In [ ]:
# plot results
alternative_benchmarking_df = pd.read_csv('../data/alternative_mistic_benchmarks.csv')

# reformat names
alt_to_plot = alternative_benchmarking_df[['variant', 'pos_mae', 'neg_mae']].set_index('variant').melt(
    var_name='Metric', value_name='MAE', ignore_index=False
).reset_index().replace(
    {
        'mistic': 'MisTIC', 'low_spillover': 'Low spillover', 'high_spillover': 'High spillover', 'resolvi_calibrated': 'Recalibrated',
        'pos_mae': 'PosCtrl', 'neg_mae': 'NegCtrl'
    }
)

sns.barplot(
    alt_to_plot,
    x='variant', y='MAE', hue='Metric'
)
plt.xlabel('MisTIC simulation variant')
plt.savefig('../results/mistic_simulation_variants.png', bbox_inches='tight')